# Poker Equity and Expected Value Analysis

In this project, I want to dive into how to make the optimal decision between calling and folding in poker. This decision is usually made with a gut feeling, or maybe the knowledge that you have a straight and your opponent probably doesn't have one. 

I want to use math and probability to quantify aspects of the decision making process and to figure out the optimal play under uncertainty. 

Let's start off by assigning a value to the strength of a given 5 card hand.

In [139]:
import random as rd
from itertools import combinations

ranks = '23456789TJQKA'
suits = 'cdhs'

rank_values = {}
for i in range(2,15):
    rank_values[ranks[i-2]] = i

hand_ranks = {
    "High card": 0,
    "Pair": 1,
    "Two pair": 2,
    "Three of a kind": 3,
    "Straight": 4,
    "Flush": 5,
    "Full house": 6,
    "Four of a kind": 7,
    "Straight flush": 8,
    "Royal flush": 9
}

def create_deck():
    deck = []

    for rank in ranks:
        for suit in suits:
            deck.append((rank,suit))
    
    return deck


def evaluate_hands(hands, board):
    
    all_hand_types = []
    all_scores = []

    for hand in hands:
        cards = hand + board

        rank_counts = {}
        suit_counts = {}
    
        for card in cards:
            rank = card[0]
            suit = card[1]

            if rank in rank_counts:
                rank_counts[rank] += 1
            else:
                rank_counts[rank] = 1

            if suit in suit_counts:
                suit_counts[suit] += 1
            else:
                suit_counts[suit] = 1

        pairs = [x for x in rank_counts if rank_counts[x] == 2]
        num_pairs = len(pairs)

        trips = [x for x in rank_counts if rank_counts[x] == 3]
        num_trips = len(trips)

        quads = [x for x in rank_counts if rank_counts[x] == 4]
        num_quads = len(quads)

        # check for flush
        flush = False
        flush_suit = ''

        for n in suit_counts:
            if suit_counts[n] >= 5:
                flush = True
                flush_suit = n
                break

        # check for straight
        rank_numbers = [rank_values[card[0]] for card in cards]
        unique_ranks = sorted(set(rank_numbers))

        if 14 in unique_ranks:
            unique_ranks.insert(0,1)

        straight = False

        for n in range(len(unique_ranks) - 4): 
            if unique_ranks[n+4] - unique_ranks[n] == 4:
                straight = True
                straight_high = unique_ranks[n+4]
                break

        # check for straight flush
        straight_flush = False
        royal_flush = False

        if flush:
            flush_cards = [card for card in cards if card[1] == flush_suit]

            flush_rank_numbers = [rank_values[card[0]] for card in flush_cards]
            unique_flush_ranks = sorted(set(flush_rank_numbers))

            if 14 in unique_flush_ranks:
                unique_flush_ranks.insert(0,1)
            
            for i in range(len(unique_flush_ranks) - 4):
                if unique_flush_ranks[i+4] - unique_flush_ranks[i] == 4:
                    straight_flush = True
                    straight_flush_high = unique_flush_ranks[i+4]
                    break

            if all(rank in flush_rank_numbers for rank in [10, 11, 12, 13, 14]):
                royal_flush = True

        if royal_flush:
            hand_type = 'Royal flush'

            score = (9,)

        elif straight_flush:
            hand_type = "Straight flush"

            score = (8, straight_flush_high)

        elif num_quads == 1:
            hand_type = "Four of a kind"

            kicker = max(
                [rank_values[card[0]] for card in cards 
                if card[0] not in quads]
            )
            
            score = (7, rank_values[quads[0]], kicker)
        
        elif num_trips >= 2:
            hand_type = 'Full house'

            trip_ranks = sorted(
                [rank_values[x] for x in trips],
                reverse=True)

            score = (6, trip_ranks[0], trip_ranks[1])

        elif num_trips == 1 and num_pairs == 1:
            hand_type = "Full house"

            trips_rank = rank_values[trips[0]]
            pairs_rank = max(
                [rank_values[card[0]]
                for card in cards 
                if card[0] in pairs])

            score = (6, trips_rank, pairs_rank)

        elif flush:
            hand_type = 'Flush'

            flush_cards = sorted(
                [rank_values[card[0]] 
                for card in cards 
                if card[1] == flush_suit],
                reverse=True)
            
            score = (5, *flush_cards[:5])

            
        
        elif straight:
            hand_type = 'Straight'

            score = (4, straight_high)

        elif num_trips == 1:
            hand_type = "Three of a kind"

            trip_rank = rank_values[trips[0]]

            kickers = sorted([rank_values[card[0]]
                            for card in cards 
                            if card[0] not in trips]
                            , reverse=True)
            
            score = (3, trip_rank, *kickers[:2])

        elif num_pairs >= 2:
            hand_type = "Two pair"

            pairs_ranks = sorted([rank_values[x] for x in pairs], reverse=True)[:2]
            kicker = max(
                rank_values[card[0]] 
                for card in cards 
                if card[0] not in pairs)

            score = (2, *pairs_ranks, kicker)


        elif num_pairs == 1:
            hand_type = "Pair"
            pair_rank = rank_values[pairs[0]]

            kickers = [
                rank_values[card[0]]
                for card in cards
                if card[0] != pairs[0]
            ]

            kickers = sorted(kickers, reverse=True)

            score = (1, pair_rank, *kickers[:3])

        else:
            hand_type = "High card"

            sorted_ranks = sorted(rank_numbers, reverse=True)

            score = (0, *sorted_ranks[:5])

        all_hand_types.append(hand_type)
        all_scores.append(score)
        
    return hands, board, all_hand_types, all_scores

The value of the person's best 5 card hand is represented as a tuple. With this format, hands can easily be compared element by element to determine the stronger hand. To refer to the person we're focused on, we will use the term hero, and the opponents will be villains.

In [176]:
hero = [[('9','h'), ('T','h')]]
villain = [[('8','s'), ('2','c')]]

hands = hero + villain

board = [('2','d'), ('4','s'),('A','h'), ('9','c'),('6','c')]


def determine_winner(all_scores):
    highest = max(all_scores)
    winners = []

    for i in range(len(all_scores)):
        if all_scores[i] == highest:
            winners.append(i+1)
    
    return winners

_,_,all_hand_types, all_scores = evaluate_hands(hands, board)

print(hands)
print(all_hand_types)
print(all_scores)
print('Winner(s):',determine_winner(all_scores))

[[('9', 'h'), ('T', 'h')], [('8', 's'), ('2', 'c')]]
['Pair', 'Pair']
[(1, 9, 14, 10, 6), (1, 2, 14, 9, 8)]
Winner(s): [1]


**Exact Equity**

With the knowledge of the players' hands and the other known cards on the board, we can create every combination of the remaining unknown cards in the deck.

In Texas Hold'em, each player has 2 cards and the final board has 5 community card. Each player has 7 cards to create the best 5 card hand they can.

If there are (n) unknown cards remaining in the deck and (k) community cards to be dealt, the number of possible board runouts is:

$$
\binom{n}{k} = \frac{n!}{k!(n-k)!}
$$

For example, if the flop is known, there are 47 unknown cards remaining considering one opponent's hand. Two more cards still need to dealt, giving:

$$
\binom{47}{2} = \frac{47!}{2!(47-2)!} = \text{1,081}
$$
possible turn and river combinations. We can use every combination to calculate the exact equity of a given hand.

$$
E = \frac{W + \frac{1}{2}T}{\text{runouts}}
$$

Where:
- $W$ number of wins
- $T$ number of ties (multiplied by 1/2 assuming we have a two-way tie)

In [141]:
def remaining_deck(known_cards):
    deck = create_deck()

    for card in known_cards:
        deck.remove(card)

    return deck

def poker_equity(hero, villains, board):
    all_hands = [hero] + villains

    known_cards = board.copy()

    for hand in all_hands:
        known_cards += hand

    cards_needed = 5 - len(board)

    deck = remaining_deck(known_cards)
    runouts = list(combinations(deck, cards_needed))

    hero_wins = 0
    hero_share = 0

    for runout in runouts:
        new_board = board + list(runout)
        
        _, _, all_hand_types, all_scores = evaluate_hands(all_hands, new_board)

        winners = determine_winner(all_scores)

        if 1 in winners:
            if len(winners) == 1:
                hero_wins += 1
            else:
                hero_share += 1 / len(winners)

    equity = (hero_wins + hero_share) / len(runouts)

    return equity


Just for fun, let's compare the exact equity calculator with a monte carlo equity calculator. As the number of simulations increases, we can expect the estimate to become more reliable.

In [206]:
def monte_carlo_equity(hero, villains, board, simulations):
    all_hands = [hero] + villains

    hero_wins = 0
    hero_share = 0

    for _ in range(simulations):

        known_cards = board.copy()

        for hand in all_hands:
            known_cards += hand

        deck = remaining_deck(known_cards)
        rd.shuffle(deck)

        cards_needed = 5 - len(board)

        runout = deck[:cards_needed]

        new_board = board + runout

        _, _, all_hand_types, all_scores = evaluate_hands(all_hands, new_board)

        winners = determine_winner(all_scores)

        if 1 in winners:
            if len(winners) == 1:
                hero_wins += 1
            else:
                hero_share += 1/len(winners)

    equity = (hero_wins + hero_share) / simulations

    return equity

def percent_error(theoretical, observed):
    percent_error = abs((theoretical - observed) / theoretical)

    return round(percent_error*100,2)

hero = [('A', 's'), ('A', 'c')]

villain = [[('Q', 'h'), ('Q', 'd')]]

board = [('8', 's'), ('3', 's'), ('9', 'c')]

theoretical = poker_equity(hero, villain, board)
print("exact hero equity:", round(theoretical*100,2), '%\n')

sims1 = 100

observed = monte_carlo_equity(hero, villain, board, sims1)
print(f"monte carlo hero equity ({sims1} sims):", observed*100, '%')

percent_error1 = percent_error(theoretical, observed)
print("percent error:", percent_error1,'%\n')


sims2 = 10000

observed = monte_carlo_equity(hero, villain, board, sims2)
print(f"monte carlo hero equity ({sims2} sims):", round(observed*100,2), '%')

percent_error2 = percent_error(theoretical, observed)
print("percent error:", percent_error2, '%')





exact hero equity: 90.91 %

monte carlo hero equity (100 sims): 97.0 %
percent error: 6.7 %

monte carlo hero equity (10000 sims): 90.28 %
percent error: 0.69 %


Theres one big problem with applying equity calculations, we don't know the other person's hand! In this situation, the best we can do is guess what hands the other person may have based on their play style and mannerisms, and assign weights to the range of hands we think they have. First, lets make it easier to create the hands they might have.

In [143]:
def generate_pair(hand):
    suits = 'cdsh'

    cards = []

    for suit in suits:
        cards.append((hand[0], suit))

    pairs = [list(pair) for pair in combinations(cards, 2)]

    return pairs

def generate_suited(rank1, rank2):
    suits = 'cdhs'

    hands = []

    for suit in suits:
        hands.append([(rank1, suit), (rank2, suit)])

    return hands

def generate_offsuit(rank1, rank2):
    suits = 'cdsh'

    hands = []

    for suit1 in suits:
        for suit2 in suits:
            
            if suit1 != suit2:
                hands.append([(rank1, suit1), (rank2, suit2)])
    
    return hands

def generate_hands(hand):
    if hand[0] == hand[1]:
        return generate_pair(hand[0])

    elif hand[2] == 's':
        return generate_suited(hand[0], hand[1])

    elif hand[2] == 'o':
        return generate_offsuit(hand[0], hand[1])


def generate_range(range_list, known_cards):

    all_hands = []

    for hand in range_list:
        hands = generate_hands(hand)

        hands = remove_known_cards(hands, known_cards)    

        all_hands += hands

    return all_hands

def generate_weighted_range(weighted_range, known_cards):

    all_hands = []

    for hand, weight in weighted_range.items():
        hands = generate_hands(hand)

        hands = remove_known_cards(hands, known_cards)

        for combination in hands:
            all_hands.append((combination, weight))

    return all_hands
                   

Now that we have built the possible range of hands, let's assign a weight to each hand based on how often we think they play it. To calculate our equity vs. the range of hands, we can use:

$$
E_{\text{range}} = \frac{1}{n}\sum_{i=1}^{n} E_i
$$

and for the weighted version of the range we use:

$$
E_{\text{weighted}} = \frac{\Sigma_{i=1}^{n}w_iE_i}{\Sigma_{i=1}^{n}w_i}
$$

where:
- $E_i$: equity against the i-th hand combination
- $w_i$: weight assigned to the i-th hand combination
- $n$  : number of valid hand combinations in the range

The weighted calculation gives more emphasis to the hands we think they play more often. This makes the result lean toward how we expect the oponent to play.

In [144]:

def range_equity(hero, villain_range, board):
    total_equity = 0
    
    for villain_hand in villain_range:
        equity = poker_equity(hero, [villain_hand], board)

        total_equity += equity

    
    equity = total_equity / len(villain_range)

    return equity

def weighted_range_equity(hero, weighted_hands, board):
    total_equity = 0
    total_weight = 0

    for villain_hand, weight in weighted_hands:
        equity = poker_equity(hero, [villain_hand], board)

        total_equity += equity * weight
        total_weight += weight

    return total_equity / total_weight
    


In [145]:
def remove_known_cards(hands, known_cards):

    valid_hands = []

    for hand in hands:

        if all(card not in known_cards for card in hand):
            valid_hands.append(hand)

    return valid_hands

Let's now use all the information we've gathered to make a decision on whether we should call or fold. In poker, the goal is to make money (and have fun), so we should make the decision that maximizes our expected profit over time. First, lets find the equity we must have to break even given a specific pot and call price.

$$
E_{\text{break-even}} = \frac{C}{P_{\text{total}} + C}
$$
Where:
- $C$ is the call price
- $P_{\text{total}}$ is the pot value before you call
- $E_{\text{break-even}}$ is the equity needed to break even over time

Now lets use our equity against our oponent's range to see how much money we'd make repeating the same situation

$$
EV_{\text{call}} =
E(P_{\text{total}}) - (1-E)C
$$

Where:
- $EV_{\text{call}}$ amount of money we'd make calling
- $E$ our equity against the opponent's range
- $P_{\text{total}}$ pot available before we call
- $C$ amount we call


The first term represents our expected winnings, while the second term represents our expected loss when we don't win the pot. If our calculated $EV_{\text{call}}$ is positive then, on average, we make money in this interaction, but if our $EV_{\text{call}}$ is negative we should fold. If $EV_{\text{call}} = 0$ then both calling and folding have the same expected value.

Repeating this situation, the average net profit should approach this expected value, assuming the equity and pot conditions remain constant.

In [146]:
def analyse_decision(hero, villain_range, board, pot, bet, call):
    known_cards = hero + board
    weighted_range = generate_weighted_range(villain_range, known_cards)

    equity = weighted_range_equity(hero, weighted_range, board)

    pot_after_bet = pot + bet

    break_even = call / (pot_after_bet + call)

    ev = equity * pot_after_bet - (1 - equity) * call

    if equity > break_even:
        decision = "call"
    elif equity < break_even:
        decision = 'fold'
    else:
        decision = "break_even"

    return equity, break_even, ev, decision


In [207]:
hero = [('A', 's'), ('K', 's')]

villain_range = {
    'QQ': 1.0,
    'JJ': 1.0,
    'TT': 0.5,
    'AKs': 0.5,
    'AQs': 0.25
}

board = [
    ('K', 'h'),
    ('7', 'c'),
    ('2', 'd')
]

pot = 100
bet = 25
call = 25

equity, break_even, ev, decision = analyse_decision(hero, villain_range, board, pot, bet, call)

print('poker equity:', round(equity*100,2),'%')
print('break even equity:', round(break_even*100,2),'%')
print('expected value:', round(ev,2),'$')
print('decision:', decision)

poker equity: 88.73 %
break even equity: 16.67 %
expected value: 108.09 $
decision: call


**Conclusion**

This poker model uses hand evaluation, weighted-range equity, and pot odds to determine whether calling is profitable under simplifying assumptions.

We use our expected value of calling to see if we would make or lose money on average by making the same call repeatedly. A positive EV means the call is profitable in the long run, while a negative EV means we would lose money on average.

The decision depends on our equity, the size of the pot, and how much we have to call. Even if we don't win every hand, making positive EV decisions is all we can do, and is the best way to be profitable in the long run.